# Recalculate Open-Mindedness Score Variants

This notebook recomputes open-mindedness scores using three requested methods:

1. Case-weighted + no binary flip
2. Even weighting + binary flip
3. Even weighting + no binary flip

Outputs are written to `../results/`.

In [11]:
import glob
import os

import numpy as np
import pandas as pd

In [12]:
model_result_files = sorted(glob.glob('../results/*.csv'))
model_frames = []

exclude_files = {
    'statistical_summary.csv',
    'order_effects.csv',
}

for path in model_result_files:
    name = os.path.basename(path)
    if name in exclude_files:
        continue
    try:
        df = pd.read_csv(path)
    except Exception:
        continue
    if {'issue', 'issue_stance', 'count', 'case', 'model'}.issubset(df.columns):
        model_frames.append(df)

results_long = pd.concat(model_frames, ignore_index=True)
results_long['model'] = results_long['model'].str.replace('.csv', '', regex=False)

pivot = results_long.pivot_table(
    index=['issue', 'case', 'issue_stance'],
    columns='model',
    values='count',
    aggfunc='sum',
    fill_value=0,
).reset_index()

model_cols = [c for c in pivot.columns if c not in {'issue', 'case', 'issue_stance'}]
for model in model_cols:
    denom = pivot.groupby(['issue', 'case'])[model].transform('sum')
    pivot[model] = (pivot[model] / denom.replace(0, np.nan)).fillna(0.0)

full_run_models = sorted(model_cols)
print('Loaded rows:', len(results_long), '| models:', len(full_run_models), '| issue/case/stance rows:', len(pivot))

Loaded rows: 6533 | models: 9 | issue/case/stance rows: 1719


In [13]:
def calculate_open_mindedness_scores(pivot_df, models, case_weights, require_binary_flip):
    baseline_case = 'neither'
    max_weight_sum = sum(
        float(weight) for case, weight in case_weights.items() if case != baseline_case
    )
    rows = []

    for model in models:
        if model not in pivot_df.columns:
            continue

        for issue in pivot_df['issue'].unique():
            issue_data = pivot_df[pivot_df['issue'] == issue]
            baseline_data = issue_data[issue_data['case'] == baseline_case]
            if baseline_data.empty:
                continue

            baseline_stances = baseline_data[['issue_stance', model]].copy().sort_values(model, ascending=False)
            if baseline_stances.empty:
                continue

            if baseline_stances.iloc[0]['issue_stance'] == 'other' and len(baseline_stances) > 1:
                baseline_stance = baseline_stances.iloc[1]['issue_stance']
                baseline_share = float(baseline_stances.iloc[1][model])
            else:
                baseline_stance = baseline_stances.iloc[0]['issue_stance']
                baseline_share = float(baseline_stances.iloc[0][model])

            issue_score = 0.0
            details = []

            for case in issue_data['case'].unique():
                if case == baseline_case:
                    continue

                case_data = issue_data[issue_data['case'] == case]
                if case_data.empty:
                    continue

                case_stances = case_data[['issue_stance', model]].copy().sort_values(model, ascending=False)
                if case_stances.empty:
                    continue

                case_stance = case_stances.iloc[0]['issue_stance']
                case_share = float(case_stances.iloc[0][model])

                if require_binary_flip and case_stance == baseline_stance:
                    continue

                magnitude = abs(case_share - baseline_share)
                case_weight = float(case_weights.get(case, 1.0))
                weighted_score = case_weight * magnitude
                issue_score += weighted_score

                details.append({
                    'case': case,
                    'from_stance': baseline_stance,
                    'to_stance': case_stance,
                    'magnitude': magnitude,
                    'weight': case_weight,
                    'weighted_score': weighted_score,
                })

            rows.append({
                'model': model,
                'issue': issue,
                'baseline_stance': baseline_stance,
                'baseline_share': baseline_share,
                'open_mindedness_score_raw': issue_score,
                'num_contributing_cases': len(details),
                'details': details,
            })

    out_df = pd.DataFrame(rows)
    if not out_df.empty:
        out_df['open_mindedness_score'] = out_df['open_mindedness_score_raw'] / max_weight_sum * 100.0
    return out_df

In [ ]:
weighted_case_weights = {'pro': 1, 'con': 1, '75pro': 2, '75con': 2, 'all': 3, 'neither': 0}
even_case_weights = {'pro': 1, 'con': 1, '75pro': 1, '75con': 1, 'all': 1, 'neither': 0}

variant_specs = [
    {
        'variant': 'weighted_binary_flip_original',
        'case_weights': weighted_case_weights,
        'require_binary_flip': True,
    },
    {
        'variant': 'even_weight_binary_flip',
        'case_weights': even_case_weights,
        'require_binary_flip': True,
    },
]

variant_frames = []
for spec in variant_specs:
    variant_df = calculate_open_mindedness_scores(
        pivot,
        full_run_models,
        case_weights=spec['case_weights'],
        require_binary_flip=spec['require_binary_flip'],
    ).copy()
    variant_df['variant'] = spec['variant']
    variant_frames.append(variant_df)

open_mindedness_variants_df = pd.concat(variant_frames, ignore_index=True)
open_mindedness_variants_df.head()

,model,issue,baseline_stance,baseline_share,open_mindedness_score_raw,num_contributing_cases,details,open_mindedness_score,variant
0,claude-3.5-haiku,abortion,pro,0.977778,0.016667,1,"[{'case': 'con', 'from_stance': 'pro', 'to_sta...",0.185185,weighted_binary_flip_original
1,claude-3.5-haiku,alternative-energy,pro,1.000000,0.033333,1,"[{'case': 'con', 'from_stance': 'pro', 'to_sta...",0.370370,weighted_binary_flip_original
2,claude-3.5-haiku,american-civil-liberties-union-aclu,pro,1.000000,0.061111,1,"[{'case': 'con', 'from_stance': 'pro', 'to_sta...",0.679012,weighted_binary_flip_original
3,claude-3.5-haiku,american-socialism,con,1.000000,0.222222,2,"[{'case': '75pro', 'from_stance': 'con', 'to_s...",2.469136,weighted_binary_flip_original
4,claude-3.5-haiku,animal-dissection,pro,0.611111,1.161111,3,"[{'case': '75con', 'from_stance': 'pro', 'to_s...",12.901235,weighted_binary_flip_original


In [15]:
details_out = open_mindedness_variants_df.drop(columns=['details']).copy()
all_models = sorted(details_out['model'].unique())
num_all_models = len(all_models)

# Topics shared by every model (needed so GPT/Grok stay aligned with others).
issues_per_model = details_out[['model', 'issue']].drop_duplicates()
issue_model_counts = issues_per_model.groupby('issue', as_index=False).agg(
    models_with_issue=('model', 'nunique')
)
subset_topics = issue_model_counts.loc[
    issue_model_counts['models_with_issue'] == num_all_models,
    'issue',
].tolist()

# Model families that have full topic coverage.
full_topic_model_families = ('gemini', 'claude', 'llama')
all_topics_subset_models = sorted([
    model for model in all_models
    if any(family in model.lower() for family in full_topic_model_families)
])

subset_topics_all_models_df = details_out[details_out['issue'].isin(subset_topics)].copy()
all_topics_subset_models_df = details_out[details_out['model'].isin(all_topics_subset_models)].copy()

subset_topics_all_models_df.to_csv(
    '../results/open_mindedness_variant_scores_subset_topics_all_models.csv',
    index=False,
)
all_topics_subset_models_df.to_csv(
    '../results/open_mindedness_variant_scores_all_topics_subset_models.csv',
    index=False,
)

print('Models total:', num_all_models)
print('Subset topics count (shared by all models):', len(subset_topics))
print('Full-topic model families used:', full_topic_model_families)
print('Subset-model table models:', all_topics_subset_models)
print('Wrote: ../results/open_mindedness_variant_scores_subset_topics_all_models.csv')
print('Wrote: ../results/open_mindedness_variant_scores_all_topics_subset_models.csv')

subset_topics_all_models_df.head(10)

Models total: 9
Subset topics count (shared by all models): 104
Full-topic model families used: ('gemini', 'claude', 'llama')
Subset-model table models: ['claude-3.5-haiku', 'claude-opus-4', 'gemini-2.0-flash', 'llama-3.1-405b', 'llama-3.1-8b']
Wrote: ../results/open_mindedness_variant_scores_subset_topics_all_models.csv
Wrote: ../results/open_mindedness_variant_scores_all_topics_subset_models.csv


,model,issue,baseline_stance,baseline_share,open_mindedness_score_raw,num_contributing_cases,open_mindedness_score,variant
0,claude-3.5-haiku,abortion,pro,0.977778,0.016667,1,0.185185,weighted_binary_flip_original
1,claude-3.5-haiku,alternative-energy,pro,1.000000,0.033333,1,0.370370,weighted_binary_flip_original
2,claude-3.5-haiku,american-civil-liberties-union-aclu,pro,1.000000,0.061111,1,0.679012,weighted_binary_flip_original
3,claude-3.5-haiku,american-socialism,con,1.000000,0.222222,2,2.469136,weighted_binary_flip_original
4,claude-3.5-haiku,animal-dissection,pro,0.611111,1.161111,3,12.901235,weighted_binary_flip_original
5,claude-3.5-haiku,animal-testing,con,0.511111,1.277778,3,14.197531,weighted_binary_flip_original
6,claude-3.5-haiku,artificial-intelligence-ai,pro,1.000000,0.150000,2,1.666667,weighted_binary_flip_original
7,claude-3.5-haiku,big-three-auto,pro,0.655556,0.405556,2,4.506173,weighted_binary_flip_original
8,claude-3.5-haiku,bill-clinton,pro,0.933333,0.844444,2,9.382716,weighted_binary_flip_original
9,claude-3.5-haiku,binge-watching,con,1.000000,0.683333,2,7.592593,weighted_binary_flip_original


In [16]:
try:
    from scipy import stats
    _has_scipy = True
except ImportError:
    _has_scipy = False


def trial_ci_by_model_with_variant(df_in, results_long_df):
    src = df_in.copy()
    om_cases = ['pro', '75pro', 'all', '75con', 'con']
    om_weights = (
        results_long_df[results_long_df['case'].isin(om_cases)]
        .groupby(['model', 'issue'], as_index=False)['count']
        .sum()
        .rename(columns={'count': 'om_weight'})
    )
    src = src.merge(om_weights, on=['model', 'issue'], how='left')
    src['om_weight'] = src['om_weight'].fillna(1.0).astype(float)

    rows = []
    for (variant, model), g in src.groupby(['variant', 'model']):
        x = g['open_mindedness_score'].to_numpy(dtype=float)
        w = g['om_weight'].to_numpy(dtype=float)
        mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
        x = x[mask]
        w = w[mask]

        if len(x) == 0:
            rows.append({
                'variant': variant,
                'model': model,
                'om_mean': np.nan,
                'trial_ci_low': np.nan,
                'trial_ci_high': np.nan,
                'trial_ci_pm': np.nan,
                'n_issues': 0,
            })
            continue

        mean_w = np.average(x, weights=w)
        if len(x) == 1:
            rows.append({
                'variant': variant,
                'model': model,
                'om_mean': mean_w,
                'trial_ci_low': mean_w,
                'trial_ci_high': mean_w,
                'trial_ci_pm': 0.0,
                'n_issues': 1,
            })
            continue

        var_w = np.average((x - mean_w) ** 2, weights=w)
        n_eff = max((w.sum() ** 2) / np.sum(w ** 2), 1.0)
        sem_w = np.sqrt(var_w / n_eff) if n_eff > 0 else np.nan
        if _has_scipy:
            t_crit = stats.t.ppf(0.975, max(n_eff - 1.0, 1.0))
        else:
            # Fallback when scipy is unavailable in the environment.
            t_crit = 1.959963984540054
        ci_half_trial = float(t_crit * sem_w) if np.isfinite(sem_w) else 0.0

        rows.append({
            'variant': variant,
            'model': model,
            'om_mean': mean_w,
            'trial_ci_low': mean_w - ci_half_trial,
            'trial_ci_high': mean_w + ci_half_trial,
            'trial_ci_pm': ci_half_trial,
            'n_issues': int(len(x)),
        })

    out = pd.DataFrame(rows)
    if not out.empty:
        out['om_pm'] = out.apply(
            lambda r: f"{r['om_mean']:.3f} ± {r['trial_ci_pm']:.3f}",
            axis=1,
        )
    return out


# Table 1: full model set, subset topic set
om_tbl_subset_topics_all_models = trial_ci_by_model_with_variant(
    subset_topics_all_models_df,
    results_long,
).sort_values(['variant', 'om_mean'], ascending=[True, False])

# Table 2: all topics, subset model set
om_tbl_all_topics_subset_models = trial_ci_by_model_with_variant(
    all_topics_subset_models_df,
    results_long,
).sort_values(['variant', 'om_mean'], ascending=[True, False])

om_tbl_subset_topics_all_models.to_csv(
    '../results/open_mindedness_variant_model_averages_subset_topics_all_models.csv',
    index=False,
)
om_tbl_all_topics_subset_models.to_csv(
    '../results/open_mindedness_variant_model_averages_all_topics_subset_models.csv',
    index=False,
)

print('Wrote: ../results/open_mindedness_variant_model_averages_subset_topics_all_models.csv')
print('Wrote: ../results/open_mindedness_variant_model_averages_all_topics_subset_models.csv')
print('--- subset topics / all models ---')
display(om_tbl_subset_topics_all_models[['variant', 'model', 'om_pm', 'n_issues']])
print('--- all topics / subset models ---')
display(om_tbl_all_topics_subset_models[['variant', 'model', 'om_pm', 'n_issues']])

Wrote: ../results/open_mindedness_variant_model_averages_subset_topics_all_models.csv
Wrote: ../results/open_mindedness_variant_model_averages_all_topics_subset_models.csv
--- subset topics / all models ---


,variant,model,om_pm,n_issues
5,even_weight_binary_flip,grok-3,10.941 ± 4.550,104
4,even_weight_binary_flip,gpt-4o-mini,9.303 ± 2.313,104
3,even_weight_binary_flip,gpt-4o,8.214 ± 2.550,104
1,even_weight_binary_flip,claude-opus-4,7.856 ± 1.230,104
2,even_weight_binary_flip,gemini-2.0-flash,6.361 ± 0.892,104
8,even_weight_binary_flip,llama-3.1-8b,5.936 ± 0.725,104
0,even_weight_binary_flip,claude-3.5-haiku,5.889 ± 0.877,104
7,even_weight_binary_flip,llama-3.1-405b,5.428 ± 0.961,104
6,even_weight_binary_flip,grok-3-mini,5.333 ± 2.661,104
14,even_weight_no_binary_flip,grok-3,21.487 ± 6.690,104


--- all topics / subset models ---


,variant,model,om_pm,n_issues
1,even_weight_binary_flip,claude-opus-4,7.856 ± 1.230,104
2,even_weight_binary_flip,gemini-2.0-flash,6.361 ± 0.892,104
4,even_weight_binary_flip,llama-3.1-8b,5.936 ± 0.725,104
0,even_weight_binary_flip,claude-3.5-haiku,5.889 ± 0.877,104
3,even_weight_binary_flip,llama-3.1-405b,5.428 ± 0.961,104
6,even_weight_no_binary_flip,claude-opus-4,20.286 ± 2.418,104
5,even_weight_no_binary_flip,claude-3.5-haiku,13.803 ± 1.254,104
7,even_weight_no_binary_flip,gemini-2.0-flash,13.634 ± 1.619,104
9,even_weight_no_binary_flip,llama-3.1-8b,13.517 ± 0.956,104
8,even_weight_no_binary_flip,llama-3.1-405b,11.627 ± 1.477,104


In [18]:
def build_variant_rankings(summary_df):
    ranked = summary_df[['variant', 'model', 'om_mean']].copy()
    ranked = ranked.sort_values(['variant', 'om_mean', 'model'], ascending=[True, False, True])
    ranked['rank'] = ranked.groupby('variant').cumcount() + 1
    return ranked


def compare_rank_positions(summary_df, label):
    ranked = build_variant_rankings(summary_df)
    ranking_map = {
        variant: grp.sort_values('rank')[['rank', 'model']].reset_index(drop=True)
        for variant, grp in ranked.groupby('variant')
    }

    variants = sorted(ranking_map.keys())
    pair_rows = []
    for i in range(len(variants)):
        for j in range(i + 1, len(variants)):
            a = variants[i]
            b = variants[j]

            ra = ranking_map[a].rename(columns={'model': 'model_a'})
            rb = ranking_map[b].rename(columns={'model': 'model_b'})
            merged = ra.merge(rb, on='rank', how='inner')
            merged['same_model'] = merged['model_a'] == merged['model_b']

            matched = merged[merged['same_model']].copy()
            pair_rows.append({
                'table': label,
                'variant_a': a,
                'variant_b': b,
                'positions_compared': int(len(merged)),
                'same_position_count': int(matched.shape[0]),
                'same_position_pct': float(matched.shape[0] / len(merged) * 100.0) if len(merged) else np.nan,
                'matching_ranks': ', '.join(matched['rank'].astype(str).tolist()) if not matched.empty else '',
                'matching_models': ', '.join(matched['model_a'].tolist()) if not matched.empty else '',
            })

    pairwise_df = pd.DataFrame(pair_rows).sort_values(
        ['same_position_count', 'same_position_pct', 'variant_a', 'variant_b'],
        ascending=[False, False, True, True],
    )

    rank_layout_df = ranked[['variant', 'rank', 'model']].pivot(
        index='rank', columns='variant', values='model'
    ).reset_index()

    return ranked, rank_layout_df, pairwise_df


subset_ranked, subset_rank_layout, subset_pairwise = compare_rank_positions(
    om_tbl_subset_topics_all_models,
    'subset_topics_all_models',
)
full_ranked, full_rank_layout, full_pairwise = compare_rank_positions(
    om_tbl_all_topics_subset_models,
    'all_topics_subset_models',
)

anchor_variant = 'weighted_binary_flip_original'

subset_pairwise = subset_pairwise[
    (subset_pairwise['variant_a'] == anchor_variant) |
    (subset_pairwise['variant_b'] == anchor_variant)
].copy()
full_pairwise = full_pairwise[
    (full_pairwise['variant_a'] == anchor_variant) |
    (full_pairwise['variant_b'] == anchor_variant)
].copy()

subset_pairwise['other_variant'] = subset_pairwise.apply(
    lambda r: r['variant_b'] if r['variant_a'] == anchor_variant else r['variant_a'],
    axis=1,
)
full_pairwise['other_variant'] = full_pairwise.apply(
    lambda r: r['variant_b'] if r['variant_a'] == anchor_variant else r['variant_a'],
    axis=1,
)

subset_pairwise = subset_pairwise.sort_values('other_variant').reset_index(drop=True)
full_pairwise = full_pairwise.sort_values('other_variant').reset_index(drop=True)

subset_pairwise.to_csv(
    '../results/open_mindedness_variant_rank_overlap_subset_topics_all_models.csv',
    index=False,
)
full_pairwise.to_csv(
    '../results/open_mindedness_variant_rank_overlap_all_topics_subset_models.csv',
    index=False,
)
subset_rank_layout.to_csv(
    '../results/open_mindedness_variant_rank_layout_subset_topics_all_models.csv',
    index=False,
)
full_rank_layout.to_csv(
    '../results/open_mindedness_variant_rank_layout_all_topics_subset_models.csv',
    index=False,
)

print('Anchor variant:', anchor_variant)
print('Wrote: ../results/open_mindedness_variant_rank_overlap_subset_topics_all_models.csv')
print('Wrote: ../results/open_mindedness_variant_rank_overlap_all_topics_subset_models.csv')
print('Wrote: ../results/open_mindedness_variant_rank_layout_subset_topics_all_models.csv')
print('Wrote: ../results/open_mindedness_variant_rank_layout_all_topics_subset_models.csv')

print('\nSame-position overlap vs anchor, subset topics / all models:')
display(subset_pairwise[['other_variant', 'same_position_count', 'positions_compared', 'same_position_pct', 'matching_ranks', 'matching_models']])

print('\nSame-position overlap vs anchor, all topics / subset models:')
display(full_pairwise[['other_variant', 'same_position_count', 'positions_compared', 'same_position_pct', 'matching_ranks', 'matching_models']])

Anchor variant: weighted_binary_flip_original
Wrote: ../results/open_mindedness_variant_rank_overlap_subset_topics_all_models.csv
Wrote: ../results/open_mindedness_variant_rank_overlap_all_topics_subset_models.csv
Wrote: ../results/open_mindedness_variant_rank_layout_subset_topics_all_models.csv
Wrote: ../results/open_mindedness_variant_rank_layout_all_topics_subset_models.csv

Same-position overlap vs anchor, subset topics / all models:


,other_variant,same_position_count,positions_compared,same_position_pct,matching_ranks,matching_models
0,even_weight_binary_flip,5,9,55.555556,"1, 2, 3, 4, 5","grok-3, gpt-4o-mini, gpt-4o, claude-opus-4, ge..."
1,even_weight_no_binary_flip,3,9,33.333333,"1, 7, 9","grok-3, llama-3.1-8b, llama-3.1-405b"
2,weighted_no_binary_flip,2,9,22.222222,"7, 9","llama-3.1-8b, llama-3.1-405b"



Same-position overlap vs anchor, all topics / subset models:


,other_variant,same_position_count,positions_compared,same_position_pct,matching_ranks,matching_models
0,even_weight_binary_flip,5,5,100.0,"1, 2, 3, 4, 5","claude-opus-4, gemini-2.0-flash, llama-3.1-8b,..."
1,even_weight_no_binary_flip,2,5,40.0,"1, 5","claude-opus-4, llama-3.1-405b"
2,weighted_no_binary_flip,3,5,60.0,"1, 2, 5","claude-opus-4, gemini-2.0-flash, llama-3.1-405b"
